# 🦅 Bird Species Classification using Deep Learning (CNN + Transfer Learning)

This notebook walks through a complete pipeline for classifying bird species using:
- A **custom CNN** built from scratch
- **Transfer Learning** with **MobileNetV2**
- Dataset: [100 Bird Species – Kaggle](https://www.kaggle.com/datasets/gpiosenka/100-bird-species)

**Sections:**
1. Environment Setup & Dataset Download
2. Data Preprocessing & Augmentation
3. Custom CNN Model
4. Transfer Learning with MobileNetV2
5. Model Training
6. Evaluation (Accuracy, Loss, Confusion Matrix)
7. Prediction on Custom Image
8. Deployment – Save Model, Export Labels, Flask API

> **Runtime:** Set to **GPU** (`Runtime > Change runtime type > T4 GPU`) for faster training.

---
## 1. Environment Setup & Dataset Download

In [ ]:
# ── Install required packages ──────────────────────────────────────────────────
!pip install -q kaggle
!pip install -q flask flask-ngrok pyngrok  # for the Flask demo at the end

In [ ]:
# ── Upload your kaggle.json API key ───────────────────────────────────────────
# Go to https://www.kaggle.com/account → 'Create New API Token' to download
# kaggle.json, then run this cell to upload it.
from google.colab import files
import os

print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Move to the expected location and restrict permissions
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("kaggle.json configured successfully!")

In [ ]:
# ── Download the 100 Bird Species dataset from Kaggle ─────────────────────────
!kaggle datasets download -d gpiosenka/100-bird-species
print("Dataset downloaded!")

In [ ]:
# ── Unzip the dataset ──────────────────────────────────────────────────────────
import zipfile

zip_path = "100-bird-species.zip"
extract_dir = "bird_species"

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

print(f"Dataset extracted to '{extract_dir}/'")

# Verify folder structure
import os
for split in ["train", "test", "valid"]:
    split_path = os.path.join(extract_dir, split)
    if os.path.isdir(split_path):
        n_classes = len(os.listdir(split_path))
        print(f"  {split}: {n_classes} classes")
    else:
        print(f"  WARNING: '{split}' folder not found at '{split_path}'")

In [ ]:
# ── Global configuration ───────────────────────────────────────────────────────
import os

BASE_DIR    = "bird_species"          # root directory of the extracted dataset
TRAIN_DIR   = os.path.join(BASE_DIR, "train")
VALID_DIR   = os.path.join(BASE_DIR, "valid")
TEST_DIR    = os.path.join(BASE_DIR, "test")

IMG_SIZE    = (224, 224)              # target image dimensions
BATCH_SIZE  = 32
EPOCHS_CNN  = 15                      # epochs for the custom CNN
EPOCHS_TL   = 10                      # epochs for the MobileNetV2 fine-tune phase
NUM_CLASSES = len(os.listdir(TRAIN_DIR))  # inferred from folder count

print(f"Image size  : {IMG_SIZE}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Num classes : {NUM_CLASSES}")

---
## 2. Data Preprocessing & Augmentation

In [ ]:
# ── Core imports ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks, optimizers

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available     : {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# ── Build ImageDataGenerators with augmentation ────────────────────────────────

# Training generator – with heavy augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,          # normalize pixel values to [0, 1]
    rotation_range=30,           # random rotation up to 30 degrees
    zoom_range=0.2,              # random zoom
    width_shift_range=0.2,       # horizontal shift
    height_shift_range=0.2,      # vertical shift
    horizontal_flip=True,        # random horizontal flip
    shear_range=0.15,            # shear transformation
    fill_mode="nearest"          # fill strategy for new pixels
)

# Validation & test generators – only normalization (no augmentation)
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

# Flow images from directories
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

valid_generator = val_test_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# Class name lookup (index → label)
class_names = list(train_generator.class_indices.keys())
print(f"Number of classes found: {len(class_names)}")
print(f"First 5 classes: {class_names[:5]}")

In [ ]:
# ── Visualize sample images from the training set ─────────────────────────────

def show_sample_images(generator, class_names, n_cols=5, n_rows=3):
    """Display a grid of sample images with their class labels."""
    images, labels = next(generator)
    n_samples = n_cols * n_rows

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    fig.suptitle("Sample Training Images", fontsize=16, fontweight="bold")

    for idx, ax in enumerate(axes.flatten()):
        if idx < len(images):
            ax.imshow(images[idx])
            label_idx = np.argmax(labels[idx])
            ax.set_title(class_names[label_idx], fontsize=8, pad=4)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_sample_images(train_generator, class_names)

---
## 3. Custom CNN Model

In [ ]:
# ── Build a custom CNN from scratch ───────────────────────────────────────────

def build_custom_cnn(input_shape, num_classes):
    """
    Custom CNN with Conv2D, MaxPooling, BatchNormalization, and Dropout layers.
    Architecture:
      3 × (Conv → BN → Conv → BN → MaxPool → Dropout)
      Flatten → Dense(512) → BN → Dropout → Dense(num_classes, softmax)
    """
    model = models.Sequential(name="Custom_CNN")

    # ── Block 1 ────────────────────────────────────────────────────────────────
    model.add(layers.Conv2D(32, (3, 3), activation="relu", padding="same",
                            input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # ── Block 2 ────────────────────────────────────────────────────────────────
    model.add(layers.Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # ── Block 3 ────────────────────────────────────────────────────────────────
    model.add(layers.Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.30))

    # ── Classifier head ────────────────────────────────────────────────────────
    model.add(layers.Flatten())
    model.add(layers.Dense(512, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation="softmax"))

    return model


cnn_model = build_custom_cnn(
    input_shape=(*IMG_SIZE, 3),
    num_classes=NUM_CLASSES
)

# Compile with Adam optimizer and categorical cross-entropy loss
cnn_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

---
## 4. Transfer Learning with MobileNetV2

In [ ]:
# ── Build MobileNetV2 transfer-learning model ──────────────────────────────────
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import Model

def build_mobilenetv2(input_shape, num_classes, freeze_base=True):
    """
    Transfer-learning model using MobileNetV2 pretrained on ImageNet.
    - Base model weights are frozen initially.
    - A custom classification head is appended.
    """
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,          # exclude ImageNet classifier
        weights="imagenet"
    )
    base_model.trainable = not freeze_base   # freeze for feature extraction

    # ── Custom head ────────────────────────────────────────────────────────────
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=base_model.input, outputs=outputs,
                  name="MobileNetV2_Transfer")
    return model, base_model


tl_model, base_model = build_mobilenetv2(
    input_shape=(*IMG_SIZE, 3),
    num_classes=NUM_CLASSES,
    freeze_base=True
)

tl_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"Total params          : {tl_model.count_params():,}")
print(f"Trainable params      : {sum(np.prod(v.shape) for v in tl_model.trainable_variables):,}")
print(f"Non-trainable params  : {sum(np.prod(v.shape) for v in tl_model.non_trainable_variables):,}")

---
## 5. Model Training

In [ ]:
# ── Training callbacks ─────────────────────────────────────────────────────────
import os

os.makedirs("models", exist_ok=True)

def get_callbacks(model_name):
    """Return a standard set of Keras callbacks."""
    return [
        # Stop early if validation accuracy stops improving
        callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        # Reduce LR on plateau
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        # Save best model
        callbacks.ModelCheckpoint(
            filepath=f"models/{model_name}_best.h5",
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1
        )
    ]

In [ ]:
# ── Train the custom CNN ───────────────────────────────────────────────────────
print("Training Custom CNN...")

cnn_history = cnn_model.fit(
    train_generator,
    epochs=EPOCHS_CNN,
    validation_data=valid_generator,
    callbacks=get_callbacks("custom_cnn"),
    verbose=1
)

# Save the final CNN model
cnn_model.save("models/custom_cnn_final.h5")
print("Custom CNN saved to models/custom_cnn_final.h5")

In [ ]:
# ── Phase 1: Train only the classification head (base frozen) ─────────────────
print("Phase 1 – Training MobileNetV2 head (base frozen)...")

tl_history_phase1 = tl_model.fit(
    train_generator,
    epochs=EPOCHS_TL,
    validation_data=valid_generator,
    callbacks=get_callbacks("mobilenetv2_phase1"),
    verbose=1
)

In [ ]:
# ── Phase 2: Fine-tune – unfreeze the top layers of the base model ─────────────

# Unfreeze the last 30 layers for fine-tuning
FINE_TUNE_FROM = len(base_model.layers) - 30
for layer in base_model.layers[FINE_TUNE_FROM:]:
    layer.trainable = True

# Recompile with a lower learning rate
tl_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"Fine-tuning from layer {FINE_TUNE_FROM} onwards")
print("Phase 2 – Fine-tuning MobileNetV2...")

tl_history_phase2 = tl_model.fit(
    train_generator,
    epochs=EPOCHS_TL,
    validation_data=valid_generator,
    callbacks=get_callbacks("mobilenetv2_phase2"),
    verbose=1
)

# Save the final transfer learning model
tl_model.save("models/mobilenetv2_final.h5")
print("MobileNetV2 model saved to models/mobilenetv2_final.h5")

---
## 6. Evaluation – Accuracy, Loss & Confusion Matrix

In [ ]:
# ── Helper: plot training history ─────────────────────────────────────────────

def plot_training_history(history, title="Training History"):
    """Plot accuracy and loss curves for a Keras history object."""
    acc     = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss    = history.history["loss"]
    val_loss= history.history["val_loss"]
    epochs  = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    # Accuracy
    ax1.plot(epochs, acc,     "b-o", label="Train Accuracy",      linewidth=2)
    ax1.plot(epochs, val_acc, "r-o", label="Validation Accuracy", linewidth=2)
    ax1.set_title("Accuracy")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.legend()
    ax1.grid(True, linestyle="--", alpha=0.6)

    # Loss
    ax2.plot(epochs, loss,     "b-o", label="Train Loss",      linewidth=2)
    ax2.plot(epochs, val_loss, "r-o", label="Validation Loss", linewidth=2)
    ax2.set_title("Loss")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend()
    ax2.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.show()


# Plot custom CNN history
plot_training_history(cnn_history, title="Custom CNN – Training History")

# Merge phase 1 + phase 2 histories for MobileNetV2
def merge_histories(h1, h2):
    """Concatenate two Keras History objects."""
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    class Merged:
        history = merged
    return Merged()

tl_history_merged = merge_histories(tl_history_phase1, tl_history_phase2)
plot_training_history(tl_history_merged, title="MobileNetV2 – Training History (Phase 1 + 2)")

In [ ]:
# ── Evaluate both models on validation and test sets ──────────────────────────

def evaluate_model(model, model_name, valid_gen, test_gen):
    """Print validation and test accuracy / loss for a given model."""
    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")

    val_loss, val_acc = model.evaluate(valid_gen, verbose=0)
    print(f"  Validation  →  Loss: {val_loss:.4f}  |  Accuracy: {val_acc:.4f}")

    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    print(f"  Test        →  Loss: {test_loss:.4f}  |  Accuracy: {test_acc:.4f}")


# Reset generators before evaluation
valid_generator.reset()
test_generator.reset()

evaluate_model(cnn_model, "Custom CNN",         valid_generator, test_generator)
evaluate_model(tl_model,  "MobileNetV2 (TL)",   valid_generator, test_generator)

In [ ]:
# ── Confusion matrix (using the best-performing model) ────────────────────────
from sklearn.metrics import confusion_matrix, classification_report

def plot_confusion_matrix(model, generator, class_names,
                          title="Confusion Matrix", max_classes=30):
    """
    Generate and visualise a confusion matrix.
    'max_classes' limits the display to the first N classes for readability.
    """
    generator.reset()
    y_pred_probs = model.predict(generator, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = generator.classes

    # Trim to first max_classes for display readability
    mask = y_true < max_classes
    y_true_sub = y_true[mask]
    y_pred_sub = y_pred[mask]
    names_sub   = class_names[:max_classes]

    cm = confusion_matrix(y_true_sub, y_pred_sub)

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
                xticklabels=names_sub, yticklabels=names_sub, ax=ax)
    ax.set_title(f"{title} (first {max_classes} classes)", fontsize=14, pad=14)
    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label",      fontsize=11)
    plt.xticks(rotation=90,  fontsize=7)
    plt.yticks(rotation=0,   fontsize=7)
    plt.tight_layout()
    plt.show()

    # Per-class metrics (full report)
    print("\nClassification Report (all classes):")
    generator.reset()
    y_pred_all = np.argmax(model.predict(generator, verbose=0), axis=1)
    print(classification_report(generator.classes, y_pred_all,
                                 target_names=class_names))


# Use MobileNetV2 (typically higher accuracy) for the confusion matrix
valid_generator.reset()
plot_confusion_matrix(tl_model, valid_generator, class_names,
                      title="MobileNetV2 – Validation Set Confusion Matrix")

---
## 7. Prediction on a Custom Image

In [ ]:
# ── Upload and classify a custom bird image ────────────────────────────────────
from google.colab import files
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def predict_bird_species(model, class_names, img_path,
                          img_size=(224, 224), top_k=5):
    """
    Load an image from disk, run inference, and display the top-K predictions.

    Parameters
    ----------
    model       : trained Keras model
    class_names : list of class label strings
    img_path    : path to the image file
    img_size    : (H, W) to resize the image to
    top_k       : number of top predictions to display
    """
    # Load & preprocess
    img = keras_image.load_img(img_path, target_size=img_size)
    img_array = keras_image.img_to_array(img) / 255.0     # normalize
    img_batch = np.expand_dims(img_array, axis=0)          # add batch dim

    # Predict
    preds = model.predict(img_batch, verbose=0)[0]         # shape: (num_classes,)
    top_indices = np.argsort(preds)[::-1][:top_k]

    # Display image with top prediction
    top_label = class_names[top_indices[0]]
    top_conf  = preds[top_indices[0]] * 100

    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 5))

    ax_img.imshow(img)
    ax_img.set_title(f"Predicted: {top_label}\nConfidence: {top_conf:.1f}%",
                     fontsize=13, fontweight="bold", pad=10)
    ax_img.axis("off")

    # Horizontal bar chart for top-K predictions
    top_labels = [class_names[i] for i in top_indices]
    top_confs  = [preds[i] * 100 for i in top_indices]
    colors = ["#2196F3" if i == 0 else "#90CAF9" for i in range(top_k)]

    bars = ax_bar.barh(top_labels[::-1], top_confs[::-1], color=colors[::-1])
    ax_bar.set_xlabel("Confidence (%)")
    ax_bar.set_title(f"Top-{top_k} Predictions", fontsize=12)
    ax_bar.set_xlim(0, 100)
    for bar, conf in zip(bars, top_confs[::-1]):
        ax_bar.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                    f"{conf:.1f}%", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f"\nTop prediction: {top_label}  ({top_conf:.2f}% confidence)")
    return top_label, top_conf


# ── Upload a bird image and run prediction ─────────────────────────────────────
print("Please upload a bird image (JPG/PNG):")
uploaded = files.upload()

for filename in uploaded.keys():
    predict_bird_species(tl_model, class_names, filename)

---
## 8. Deployment – Save Model, Export Class Labels & Flask API

In [ ]:
# ── Save the MobileNetV2 model and export class labels ─────────────────────────
import json
import os

os.makedirs("deployment", exist_ok=True)

# Save as .h5
tl_model.save("deployment/bird_species_model.h5")
print("Model saved to deployment/bird_species_model.h5")

# Save as SavedModel format (recommended for TF Serving)
tl_model.save("deployment/bird_species_savedmodel", save_format="tf")
print("SavedModel saved to deployment/bird_species_savedmodel/")

# Export class labels as JSON
label_map = {str(v): k for k, v in train_generator.class_indices.items()}
with open("deployment/class_labels.json", "w") as f:
    json.dump(label_map, f, indent=2)
print(f"Class labels exported to deployment/class_labels.json ({len(label_map)} classes)")

# Download artifacts to local machine
from google.colab import files
files.download("deployment/bird_species_model.h5")
files.download("deployment/class_labels.json")

In [ ]:
# ── Flask REST API example ─────────────────────────────────────────────────────
# This cell writes a standalone app.py and launches it via pyngrok so that
# you can send HTTP POST requests to /predict from outside Colab.
#
# Usage after the server starts:
#   curl -X POST <ngrok-url>/predict \
#        -F "file=@/path/to/bird.jpg"

flask_app_code = '''
import os
import json
import numpy as np
from flask import Flask, request, jsonify
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import io

# ── Configuration ──────────────────────────────────────────────────────────────
MODEL_PATH  = "deployment/bird_species_model.h5"
LABELS_PATH = "deployment/class_labels.json"
IMG_SIZE    = (224, 224)

# ── Load model and labels once at startup ─────────────────────────────────────
model = load_model(MODEL_PATH)
with open(LABELS_PATH) as f:
    label_map = json.load(f)   # {"0": "ALBATROSS", "1": "...", ...}

app = Flask(__name__)


@app.route("/", methods=["GET"])
def home():
    """Health-check endpoint."""
    return jsonify({"status": "ok", "message": "Bird Species API is running"})


@app.route("/predict", methods=["POST"])
def predict():
    """
    Accept a multipart/form-data POST request with a field 'file' containing
    an image, and return the top-5 predicted bird species with confidence scores.
    """
    if "file" not in request.files:
        return jsonify({"error": "No file field in request"}), 400

    file = request.files["file"]
    if file.filename == "":
        return jsonify({"error": "Empty filename"}), 400

    try:
        # Read, resize and normalize the image
        img = Image.open(io.BytesIO(file.read())).convert("RGB")
        img = img.resize(IMG_SIZE)
        img_array = np.array(img) / 255.0
        img_batch = np.expand_dims(img_array, axis=0)

        # Run inference
        preds = model.predict(img_batch, verbose=0)[0]
        top5_indices = np.argsort(preds)[::-1][:5]

        results = [
            {
                "rank": int(rank + 1),
                "species": label_map[str(idx)],
                "confidence": round(float(preds[idx]) * 100, 2)
            }
            for rank, idx in enumerate(top5_indices)
        ]

        return jsonify({
            "prediction": results[0]["species"],
            "confidence": results[0]["confidence"],
            "top_5": results
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

# Write app.py
with open("app.py", "w") as f:
    f.write(flask_app_code)

print("app.py written.")
print("Starting Flask server via pyngrok...")

# ── Launch server and expose via ngrok ────────────────────────────────────────
from pyngrok import ngrok
import subprocess, time

# Start Flask as a background subprocess
server_proc = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)  # wait for server to initialise

# Open an ngrok tunnel on port 5000
public_url = ngrok.connect(5000)
print(f"\n✅ Flask API is live at: {public_url}")
print(f"   Health check : GET  {public_url}/")
print(f"   Predict      : POST {public_url}/predict  (field: 'file')")

---
## Summary

| Step | Description | Output |
|------|-------------|--------|
| 1 | Dataset download from Kaggle | `bird_species/` directory |
| 2 | Preprocessing & augmentation | `ImageDataGenerator` pipelines |
| 3 | Custom CNN | `models/custom_cnn_final.h5` |
| 4 | MobileNetV2 Transfer Learning | `models/mobilenetv2_final.h5` |
| 5 | Training (CNN + TL) | Training curves |
| 6 | Evaluation | Accuracy/loss plots + confusion matrix |
| 7 | Custom image prediction | Predicted label with confidence bar |
| 8 | Deployment | `deployment/bird_species_model.h5`, `class_labels.json`, Flask API |

### Tips for Better Results
- Increase `EPOCHS_CNN` / `EPOCHS_TL` if accuracy is still improving.
- Experiment with `EfficientNetB0` or `ResNet50` as alternative base models.
- Use `ImageDataGenerator` with more aggressive augmentation for small datasets.
- Enable mixed-precision training (`tf.keras.mixed_precision`) for faster GPU training.